In [1]:
import tensorflow  as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [2]:
model = load_model("model.h5")

In [3]:
with open("onehot_encoder_geo.pkl", "rb") as file:
    one_hot_encoding = pickle.load(file)

In [4]:
with open("label.pkl", "rb") as file:
    label_encoder_gender = pickle.load(file)

In [5]:
with open("scaler.pkl", "rb") as file:
    scaler = pickle.load(file)

In [6]:
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 3,
    'HasCrCard': 2,
    'IsActiveMember': 4,
    'EstimatedSalary': 50000
}

In [7]:
geo_encoded = one_hot_encoding.transform([[input_data["Geography"]]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=one_hot_encoding.get_feature_names_out(["Geography"]))
geo_encoded_df

d:\Deep Learning P-1\ANN\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [8]:
# converted the input into dataframe which was in dict form
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,3,2,4,50000


In [9]:
input_df["Gender"] = label_encoder_gender.transform(input_df["Gender"])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,3,2,4,50000


In [10]:
# concatation of input_df and one_hot_encoding of geo 
input_df = pd.concat([input_df.drop("Geography", axis=1), geo_encoded_df], axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,3,2,4,50000,1.0,0.0,0.0


In [11]:
# performing scaler on input_df
input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         2.53355998,  2.83875637,  6.97676869, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [12]:
prediction = model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 242ms/step


array([[0.00125915]], dtype=float32)

In [13]:
prediction_proba = prediction[0][0]

In [14]:
prediction_proba

np.float32(0.0012591532)

In [15]:
if prediction_proba > 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is not likely to churn.
